# LLM Financial Anomaly Detection Pipeline

**Architecture:** ML model detects fraud → LLM explains *why* each flagged transaction is suspicious in plain English → Results validated against ground truth labels.

This mirrors real compliance workflows where black-box model outputs need audit-ready justification for human reviewers.

In [9]:
# 1. Install & imports

!pip install groq

import os, kagglehub
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
from groq import Groq

In [10]:
# 2. Load data

path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
df = pd.read_csv(os.path.join(path, "creditcard.csv"))

print(f"Total transactions : {len(df):,}")
print(f"Fraudulent         : {df['Class'].sum():,} ({df['Class'].mean()*100:.2f}%)")


Using Colab cache for faster access to the 'creditcardfraud' dataset.
Total transactions : 284,807
Fraudulent         : 492 (0.17%)


In [11]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [12]:
# 3. Train ML fraud detector

features = [c for c in df.columns if c not in ['Class', 'Time']]
X = df[features]
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

clf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("ML Model Performance on Test Set:")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))

ML Model Performance on Test Set:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     56864
       Fraud       0.96      0.76      0.85        98

    accuracy                           1.00     56962
   macro avg       0.98      0.88      0.92     56962
weighted avg       1.00      1.00      1.00     56962



In [13]:
# 4. Extract ML-flagged transactions for LLM explanation
# Get transactions the ML model flagged as fraud from the test set

test_df = X_test.copy()
test_df['True_Label'] = y_test.values
test_df['ML_Predicted'] = y_pred
test_df['Amount'] = df.loc[X_test.index, 'Amount'].values


# Flagged by ML model
flagged = test_df[test_df['ML_Predicted'] == 1].copy()

# Sample up to 10 for LLM analysis (API cost control)
flagged_sample = flagged.sample(min(10, len(flagged)), random_state=42)

print(f"Transactions flagged by ML model : {len(flagged)}")
print(f"Sending to LLM for explanation   : {len(flagged_sample)}")
print(f"\nGround truth breakdown of flagged transactions:")
print(flagged['True_Label'].value_counts().rename({0: 'Legitimate (FP)', 1: 'Actual Fraud (TP)'}))

Transactions flagged by ML model : 77
Sending to LLM for explanation   : 10

Ground truth breakdown of flagged transactions:
True_Label
Actual Fraud (TP)    74
Legitimate (FP)       3
Name: count, dtype: int64


In [14]:
# 5. LLM explains each flagged transaction
# Use environment variable for API key - never hardcode

from google.colab import userdata
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

# Top features driving fraud detection (from companion ML project)
top_features = ['V14', 'V10', 'V4', 'V11', 'V12', 'Amount']

results = []

for idx, row in flagged_sample.iterrows():
    feature_summary = "\n".join(
        [f"  - {f}: {row[f]:.4f}" for f in top_features if f in row]
    )

    prompt = f"""You are a financial compliance analyst reviewing a transaction flagged as potentially fraudulent by an ML model.\n\nTransaction details (anonymised PCA features — negative values indicate deviation from normal patterns):\n{feature_summary}\n\nThe average transaction amount in this dataset is £88.40. This transaction amount: £{row['Amount']:.2f}\n\nTask: In 3-4 sentences, explain what specific characteristics of this transaction suggest fraud risk,\nand what a compliance analyst should investigate first. Be specific about which feature values are suspicious and why."""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a financial fraud analyst. Be concise, specific, and structured. Focus on the feature values provided."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=200
    )

    explanation = response.choices[0].message.content
    results.append({
        'Transaction_ID': idx,
        'Amount': row['Amount'],
        'True_Label': 'FRAUD' if row['True_Label'] == 1 else 'Legitimate',
        'ML_Flag': 'Flagged',
        'LLM_Explanation': explanation
    })

    print(f"\n{'='*60}")
    print(f"Transaction {idx} | Amount: £{row['Amount']:.2f} | Ground Truth: {'FRAUD' if row['True_Label']==1 else 'Legitimate'}")
    print(f"\nLLM Explanation:\n{explanation}")


Transaction 119781 | Amount: £124.53 | Ground Truth: FRAUD

LLM Explanation:
This transaction is flagged as potentially fraudulent due to significant deviations from normal patterns in feature values, particularly V14 (-7.4952) and V12 (-7.1283), which indicate substantial anomalies in the transaction's behavior. The increased amount (£124.53) is also suspicious, exceeding the average transaction amount by £36.13. Furthermore, the deviations in V14 and V12 are correlated with a decrease in V10 (-5.5155), suggesting that this transaction may be attempting to evade detection by mimicking legitimate behavior while concealing the true anomalies. A compliance analyst should investigate the authenticity of the transaction amount and the reasons behind the deviations in V14 and V12.

Transaction 41569 | Amount: £1.00 | Ground Truth: FRAUD

LLM Explanation:
This transaction indicates a high fraud risk due to the combination of negative deviations from normal patterns on several PCA features, 

In [15]:
# 6. Evaluating pipeline end-to-end

results_df = pd.DataFrame(results)

tp = len(results_df[results_df['True_Label'] == 'FRAUD'])
fp = len(results_df[results_df['True_Label'] == 'Legitimate'])
total_flagged = len(results_df)

print("\n" + "="*60)
print("PIPELINE EVALUATION SUMMARY")
print("="*60)
print(f"Transactions sent to LLM for explanation : {total_flagged}")
print(f"True Positives  (actual fraud flagged)   : {tp}")
print(f"False Positives (legitimate, flagged)    : {fp}")
print(f"Precision of ML flags sent to LLM        : {tp/total_flagged*100:.1f}%")
print()
print("ML Model overall test set performance:")
print(f"  Precision : {precision_score(y_test, y_pred):.2f}")
print(f"  Recall    : {recall_score(y_test, y_pred):.2f}")
print(f"  F1        : {f1_score(y_test, y_pred):.2f}")
print()
print("Pipeline interpretation:")
print("  The ML model detects fraud with measurable precision/recall.")
print("  The LLM explains each flagged case in plain English for compliance review.")
print("  This combines the pattern-recognition strength of ML with the")
print("  explainability needed for audit trails — neither component alone suffices.")


PIPELINE EVALUATION SUMMARY
Transactions sent to LLM for explanation : 10
True Positives  (actual fraud flagged)   : 9
False Positives (legitimate, flagged)    : 1
Precision of ML flags sent to LLM        : 90.0%

ML Model overall test set performance:
  Precision : 0.96
  Recall    : 0.76
  F1        : 0.85

Pipeline interpretation:
  The ML model detects fraud with measurable precision/recall.
  The LLM explains each flagged case in plain English for compliance review.
  This combines the pattern-recognition strength of ML with the
  explainability needed for audit trails — neither component alone suffices.


Pipeline detected X% of actual fraud cases
LLM generated audit-ready explanations for all flagged transactions.

False positive rate: X% — compliance analyst reviews only these N cases
vs reviewing all 56,000 transactions manually

In [16]:
# 7. Exporting audit report

results_df.to_csv('audit_report.csv', index=False)
print("Audit report saved to audit_report.csv")
print("\nSample output:")
print(results_df[['Transaction_ID', 'Amount', 'True_Label', 'LLM_Explanation']].head(3).to_string())

Audit report saved to audit_report.csv

Sample output:
   Transaction_ID  Amount True_Label                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      LLM_Explanation
0          119781  124.53      FRAUD                                                                                Thi